# Dressa: Klasifikasi Atribut Pakaian (CNN dari Nol)

Notebook ini melatih model **Convolutional Neural Network (CNN)** dari nol (bobot acak, tanpa pretrained) untuk menebak tiga hal dari foto satu pakaian:

| Output (head) | Contoh kelas | Dipakai Dressa untuk |
|---|---|---|
| `item_type` | t-shirt, jeans, sneakers, bag | Kategori item (TOP/BOTTOM/...) dan filter lemari |
| `style` | casual, formal, sporty, ethnic | Kecocokan dengan agenda kalender |
| `season` | summer, winter, fall, spring | Proksi ketebalan untuk kecocokan cuaca |

Satu model dengan beberapa output seperti ini disebut **multi-task learning**. Ketiga head berbagi "mata" yang sama (backbone CNN), lalu masing-masing punya lapisan akhir sendiri.

**Alur notebook:**
1. Setup dan cek GPU
2. Unduh dan eksplorasi dataset
3. Pemetaan label, lalu split train/val/test
4. Preprocessing dan augmentasi
5. Definisi model CNN
6. Training
7. Evaluasi di test set
8. Export ke ONNX untuk dipakai `ml-engine`
9. (Opsional) Pembanding dengan transfer learning

**Sebelum mulai di Colab:** buka menu **Runtime → Change runtime type → T4 GPU**, lalu jalankan sel dari atas ke bawah (Shift+Enter). Dengan GPU, satu kali training penuh kira-kira 10–20 menit.

## 1. Setup

**PyTorch** adalah library deep learning yang dipakai di notebook ini. Konsep dasarnya:

- **Tensor**: array angka multidimensi, mirip array NumPy, tapi bisa dihitung di GPU dan bisa menghitung turunan (gradient) otomatis. Satu batch foto berbentuk tensor `(B, 3, H, W)`, artinya B gambar, 3 kanal warna (RGB), tinggi H, lebar W.
- **GPU** mengerjakan ribuan perkalian sekaligus, jadi training bisa 10–50× lebih cepat dibanding CPU. `device` menentukan tensor dan model ditaruh di mana.
- **Seed**: angka awal untuk generator acak. Seed yang tetap membuat hasil bisa diulang (reproducible), dan ini penting untuk laporan.

In [ ]:
!pip install -q kagglehub onnx onnxruntime onnxscript

In [ ]:
import json
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("PERINGATAN: tidak ada GPU. Di Colab: Runtime -> Change runtime type -> T4 GPU")

### Konfigurasi

Semua angka yang bisa kamu ubah untuk eksperimen dikumpulkan di sini. Istilah yang dipakai:

- `img_size`: ukuran sisi gambar input. Gambar di dataset ini kecil (sekitar 60×80 piksel), jadi 80 sudah cukup.
- `batch_size`: jumlah gambar yang diproses sekaligus sebelum bobot diperbarui.
- `epochs`: berapa kali model melihat **seluruh** data training.
- `lr` (learning rate): seberapa besar langkah pembaruan bobot. Terlalu besar membuat training tidak stabil, terlalu kecil membuat training lambat.
- `weight_decay`: penalti untuk bobot yang terlalu besar, salah satu cara mencegah overfitting.
- `loss_weights`: seberapa penting tiap head dalam total loss.

In [ ]:
CONFIG = {
    "seed": 42,
    "img_size": 80,
    "batch_size": 256,
    "epochs": 30,
    "lr": 3e-3,
    "weight_decay": 5e-4,
    "label_smoothing": 0.1,
    "dropout": 0.3,
    "loss_weights": {"item_type": 1.0, "style": 0.5, "season": 0.5},
    "run_pretrained_baseline": True,  # bagian 9, pembanding transfer learning
    "baseline_epochs": 10,
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
WORK_DIR = Path("/content" if IN_COLAB else "work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 2. Dataset

Dataset yang dipakai adalah [Fashion Product Images (Small)](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small) dari Kaggle. Isinya sekitar 44 ribu foto produk dari toko online Myntra, masing-masing dengan label di `styles.csv`.

Kenapa dataset ini cocok: fotonya produk dengan **latar putih**. Di Dressa, foto user juga lebih dulu dihapus background-nya lalu ditempel di latar putih, sehingga data training dan data asli di aplikasi mirip.

Dataset ini publik, jadi `kagglehub` bisa mengunduhnya tanpa login (sekitar 600 MB). Kalau muncul error autentikasi, buat API token di kaggle.com → Settings → API, lalu simpan `KAGGLE_USERNAME` dan `KAGGLE_KEY` di menu **Secrets** (ikon kunci) di Colab.

In [ ]:
import kagglehub

# DRESSA_DATA_DIR bisa diisi kalau dataset sudah ada di folder lokal / Google Drive.
DATA_DIR = Path(os.environ.get("DRESSA_DATA_DIR") or kagglehub.dataset_download("paramaggarwal/fashion-product-images-small"))
CSV_PATH = next(DATA_DIR.rglob("styles.csv"))
IMG_DIR = next(p for p in DATA_DIR.rglob("images") if p.is_dir())
print("CSV   :", CSV_PATH)
print("Images:", IMG_DIR, "-", sum(1 for _ in IMG_DIR.glob("*.jpg")), "file")

### Eksplorasi data (EDA)

Sebelum melatih model, lihat dulu datanya: berapa banyak contoh per kelas, dan seperti apa fotonya. Beberapa baris di `styles.csv` rusak (jumlah koma tidak pas), jadi baris itu dilewati (`on_bad_lines="skip"`).

In [ ]:
df = pd.read_csv(CSV_PATH, on_bad_lines="skip")
df = df[df["id"].apply(lambda i: (IMG_DIR / f"{i}.jpg").exists())].reset_index(drop=True)
print("Baris dengan gambar:", len(df))
print("\nmasterCategory:\n", df["masterCategory"].value_counts().head(8).to_string())
print("\nusage:\n", df["usage"].value_counts(dropna=False).to_string())
print("\nseason:\n", df["season"].value_counts(dropna=False).to_string())

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for ax, (_, row) in zip(axes.flat, df.sample(16, random_state=1).iterrows()):
    ax.imshow(Image.open(IMG_DIR / f"{row['id']}.jpg"))
    ax.set_title(f"{row['articleType']}\n{row['usage']}, {row['season']}", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Pemetaan label

Kolom `articleType` punya lebih dari 140 nilai. Banyak yang tidak relevan untuk lemari pakaian (jam tangan, parfum, perhiasan, pakaian dalam) atau terlalu sedikit contohnya (misalnya `Blazers` cuma 8 foto). Karena itu:

1. **Gabungkan** tipe yang mirip, misalnya `Casual Shoes` + `Sports Shoes` → `sneakers`.
2. **Buang** tipe yang tidak dipakai Dressa.

Untuk `style` dan `season`, label yang kosong atau terlalu jarang (`Party`, `Travel`, dst.) diberi nilai **-1**, artinya "tidak diketahui". Saat menghitung loss, nilai -1 diabaikan (`ignore_index=-1`). Jadi foto tanpa label style tetap bisa dipakai untuk melatih `item_type`.

**Catatan jujur untuk laporan:** `season` di dataset ini adalah musim katalog penjualan, bukan ukuran tebal-tipisnya pakaian. Label ini hanya proksi kasar untuk "warmth".

In [ ]:
ITEM_TYPE_MAP = {
    "t-shirt": ["Tshirts", "Lounge Tshirts"],
    "shirt": ["Shirts"],
    "top": ["Tops", "Tunics", "Camisoles"],
    "kurta": ["Kurtas", "Kurtis", "Kurta Sets"],
    "sweatshirt": ["Sweatshirts"],
    "sweater": ["Sweaters"],
    "jacket": ["Jackets", "Rain Jacket", "Blazers", "Waistcoat", "Nehru Jackets"],
    "jeans": ["Jeans", "Jeggings"],
    "trousers": ["Trousers", "Capris"],
    "shorts": ["Shorts", "Lounge Shorts"],
    "track pants": ["Track Pants", "Lounge Pants"],
    "leggings": ["Leggings", "Tights"],
    "skirt": ["Skirts"],
    "dress": ["Dresses", "Jumpsuit", "Rompers"],
    "sneakers": ["Casual Shoes", "Sports Shoes"],
    "formal shoes": ["Formal Shoes"],
    "heels": ["Heels"],
    "flats": ["Flats"],
    "sandals": ["Sandals", "Sports Sandals", "Flip Flops"],
    "bag": ["Handbags", "Backpacks", "Clutches", "Duffel Bag", "Laptop Bag", "Messenger Bag", "Rucksacks"],
    "cap": ["Caps", "Hat"],
    "belt": ["Belts"],
    "scarf": ["Scarves", "Stoles", "Mufflers"],
}
# Kategori Dressa (enum ItemCategory di database) untuk tiap kelas
ITEM_TYPE_TO_CATEGORY = {
    **dict.fromkeys(["t-shirt", "shirt", "top", "kurta", "sweatshirt", "sweater"], "TOP"),
    "jacket": "OUTERWEAR",
    **dict.fromkeys(["jeans", "trousers", "shorts", "track pants", "leggings", "skirt"], "BOTTOM"),
    "dress": "DRESS",
    **dict.fromkeys(["sneakers", "formal shoes", "heels", "flats", "sandals"], "FOOTWEAR"),
    **dict.fromkeys(["bag", "cap", "belt", "scarf"], "ACCESSORY"),
}
STYLE_MAP = {"Casual": "casual", "Formal": "formal", "Sports": "sporty", "Ethnic": "ethnic"}
SEASON_MAP = {"Summer": "summer", "Fall": "fall", "Winter": "winter", "Spring": "spring"}

CLASSES = {
    "item_type": list(ITEM_TYPE_MAP),
    "style": list(STYLE_MAP.values()),
    "season": list(SEASON_MAP.values()),
}
HEADS = list(CLASSES)

article_to_item = {a: k for k, articles in ITEM_TYPE_MAP.items() for a in articles}
data = df[df["articleType"].isin(article_to_item)].copy()

def to_index(values, mapping, classes):
    idx = {c: i for i, c in enumerate(classes)}
    return values.map(lambda v: idx.get(mapping.get(v), -1)).astype(int)

data["item_type"] = to_index(data["articleType"], article_to_item, CLASSES["item_type"])
data["style"] = to_index(data["usage"], STYLE_MAP, CLASSES["style"])
data["season"] = to_index(data["season"], SEASON_MAP, CLASSES["season"])
data = data.reset_index(drop=True)

print(f"Dipakai {len(data)} dari {len(df)} foto, {len(CLASSES['item_type'])} kelas item_type")
counts = data["item_type"].value_counts().sort_index()
plt.figure(figsize=(12, 3.5))
plt.bar([CLASSES["item_type"][i] for i in counts.index], counts.values)
plt.xticks(rotation=60, ha="right"); plt.ylabel("jumlah foto"); plt.title("Distribusi kelas item_type")
plt.tight_layout(); plt.show()
print("Label style hilang :", (data["style"] == -1).sum())
print("Label season hilang:", (data["season"] == -1).sum())

Distribusi kelasnya sangat **tidak seimbang** (*imbalanced*): t-shirt ribuan, skirt cuma sekitar seratus. Kalau dibiarkan, model akan "malas" dan cenderung menebak kelas besar saja. Ada dua penanganan di notebook ini:

1. **Class weight** di loss: kesalahan pada kelas kecil dihitung lebih berat.
2. **Macro-F1** sebagai metrik utama: rata-rata F1 per kelas, sehingga kelas kecil sama pentingnya dengan kelas besar. Metrik *accuracy* biasa bisa terlihat tinggi hanya karena model pandai menebak t-shirt.

### Split train / validation / test

- **Train (70%)**: data yang dipelajari model.
- **Validation (15%)**: dipakai saat training untuk memilih model terbaik dan mendeteksi overfitting.
- **Test (15%)**: **hanya dipakai sekali di akhir** untuk melaporkan hasil. Kalau test set ikut dipakai untuk memilih setelan, angkanya jadi terlalu optimis.

`stratify` menjaga proporsi tiap kelas tetap sama di ketiga bagian.

In [ ]:
train_df, rest_df = train_test_split(data, test_size=0.30, stratify=data["item_type"], random_state=CONFIG["seed"])
val_df, test_df = train_test_split(rest_df, test_size=0.50, stratify=rest_df["item_type"], random_state=CONFIG["seed"])
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

## 4. Preprocessing dan augmentasi

**Preprocessing** adalah langkah yang sama persis untuk setiap gambar, baik saat training maupun nanti di `ml-engine`:
1. Tempel gambar di kanvas putih berbentuk persegi (supaya proporsinya tidak gepeng).
2. Resize ke `img_size × img_size`.
3. Normalisasi: kurangi rata-rata lalu bagi dengan standar deviasi per kanal warna, supaya angka input berada di sekitar 0. Ini membuat training lebih stabil.

Supaya cepat, semua gambar dimuat sekali ke memori sebagai array `uint8` (sekitar 650 MB) dan disimpan ke file cache. Kalau runtime di-restart, sel ini cukup membaca cache.

In [ ]:
def preprocess(img: Image.Image, size: int) -> Image.Image:
    # Sama persis dengan yang akan dipakai ml-engine. Jangan diubah tanpa mengubah keduanya.
    if img.mode in ("RGBA", "LA", "P"):
        rgba = img.convert("RGBA")
        bg = Image.new("RGB", rgba.size, (255, 255, 255))
        bg.paste(rgba, mask=rgba.getchannel("A"))
        img = bg
    img = img.convert("RGB")
    w, h = img.size
    s = max(w, h)
    canvas = Image.new("RGB", (s, s), (255, 255, 255))
    canvas.paste(img, ((s - w) // 2, (s - h) // 2))
    return canvas.resize((size, size), Image.BILINEAR)

def load_split(split_df, name):
    cache = WORK_DIR / f"cache_{name}_{CONFIG['img_size']}_{len(split_df)}.npz"
    if cache.exists():
        return np.load(cache)["x"]
    def load(i):
        with Image.open(IMG_DIR / f"{i}.jpg") as im:
            return np.asarray(preprocess(im, CONFIG["img_size"]))
    with ThreadPoolExecutor(8) as pool:
        x = np.stack(list(pool.map(load, split_df["id"])))
    np.savez(cache, x=x)
    return x

t0 = time.time()
X = {name: load_split(d, name) for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
Y = {name: d[HEADS].to_numpy() for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
print({k: v.shape for k, v in X.items()}, f"{time.time() - t0:.0f}s")

# Mean/std dihitung dari data TRAIN saja (val/test tidak boleh "bocor" ke training)
MEAN = (X["train"].reshape(-1, 3).mean(0) / 255).tolist()
STD = (X["train"].reshape(-1, 3).std(0) / 255).tolist()
print("mean", np.round(MEAN, 3), "std", np.round(STD, 3))

**Dataset dan DataLoader** adalah cara standar PyTorch untuk menyuplai data. `Dataset` menjawab "berikan contoh ke-i", dan `DataLoader` mengambil contoh secara acak lalu menggabungkannya menjadi batch.

**Augmentasi** mengubah gambar training sedikit secara acak setiap kali dipakai: dibalik kiri-kanan, digeser, diperbesar atau diperkecil, kecerahannya diubah. Model jadi melihat variasi yang lebih banyak dan tidak sekadar menghafal foto training. Augmentasi **hanya** untuk data train, tidak untuk val/test.

Supaya CPU Colab yang lemah tidak jadi hambatan, augmentasi dikerjakan di GPU untuk satu batch sekaligus. Hue (rona warna) sengaja tidak diubah, karena warna adalah informasi penting untuk pakaian.

In [ ]:
def make_loader(name, shuffle):
    x = torch.from_numpy(X[name]).permute(0, 3, 1, 2).contiguous()  # (N,H,W,C) -> (N,C,H,W)
    y = torch.from_numpy(Y[name]).long()
    return DataLoader(TensorDataset(x, y), batch_size=CONFIG["batch_size"], shuffle=shuffle,
                      drop_last=shuffle, pin_memory=device.type == "cuda")

loaders = {"train": make_loader("train", True), "val": make_loader("val", False), "test": make_loader("test", False)}
MEAN_T = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)
STD_T = torch.tensor(STD, device=device).view(1, 3, 1, 1)

def to_input(xb_uint8, train):
    x = xb_uint8.to(device, non_blocking=True).float() / 255.0
    if train:
        x = augment(x)
    return (x - MEAN_T) / STD_T

def augment(x):
    B = x.shape[0]
    u = lambda lo, hi: torch.empty(B, device=x.device).uniform_(lo, hi)
    # 1) flip horizontal acak
    flip = torch.rand(B, device=x.device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    # 2) zoom dan geser acak; area kosong diisi putih (trik: 1-x supaya padding nol jadi putih)
    scale, tx, ty = u(0.85, 1.15), u(-0.1, 0.1), u(-0.1, 0.1)
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0] = 1 / scale; theta[:, 1, 1] = 1 / scale
    theta[:, 0, 2] = tx; theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = 1 - F.grid_sample(1 - x, grid, padding_mode="zeros", align_corners=False)
    # 3) kecerahan dan kontras acak (hue tidak diubah)
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    x = ((x - mean) * u(0.85, 1.15)[:, None, None, None] + mean) * u(0.9, 1.1)[:, None, None, None]
    return x.clamp(0, 1)

xb, _ = next(iter(loaders["train"]))
aug = augment(xb[:8].to(device).float() / 255).cpu()
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(xb[i].permute(1, 2, 0)); axes[0, i].axis("off")
    axes[1, i].imshow(aug[i].permute(1, 2, 0)); axes[1, i].axis("off")
axes[0, 0].set_title("asli", loc="left"); axes[1, 0].set_title("augmentasi", loc="left")
plt.tight_layout(); plt.show()

## 5. Model: DressaNet (CNN kecil bergaya ResNet)

**Convolution (konvolusi)** adalah filter kecil, misalnya 3×3, yang digeser ke seluruh gambar dan mendeteksi pola lokal. Lapisan awal belajar pola sederhana (tepi, garis, warna), dan lapisan yang lebih dalam menggabungkannya menjadi pola kompleks (kerah, sol sepatu, ritsleting). Bobot filternya **dipelajari** dari data.

Komponen yang dipakai:

- **BatchNorm**: menormalkan output tiap lapisan, sehingga training lebih cepat dan stabil.
- **ReLU**: fungsi aktivasi `max(0, x)`. Tanpa fungsi non-linear seperti ini, seberapa pun dalam jaringannya, hasilnya hanya setara satu operasi linear.
- **Stride 2**: filter melompat 2 piksel, sehingga ukuran peta fitur menjadi setengah. Jaringan jadi melihat area yang makin luas di lapisan dalam.
- **Residual block (ResNet)**: input blok ditambahkan langsung ke outputnya (`out = F(x) + x`). Jalan pintas ini membuat gradient mudah mengalir ke lapisan awal, sehingga jaringan yang dalam tetap bisa dilatih.
- **Global average pooling**: merata-ratakan setiap peta fitur menjadi satu angka, sehingga terbentuk satu vektor fitur per gambar.
- **Dropout**: saat training, sebagian neuron dimatikan secara acak. Model tidak bisa bergantung pada satu neuron saja, dan ini mengurangi overfitting.
- **Head**: satu lapisan `Linear` per tugas yang mengubah vektor fitur menjadi skor (**logit**) untuk setiap kelas.

Bentuk tensor untuk input 80×80: `3×80×80 → stem 32×40×40 → 64×20×20 → 128×10×10 → 256×5×5 → pooling 256 → head`.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        # Jalan pintas; perlu konvolusi 1x1 kalau ukuran/kanal berubah
        self.shortcut = nn.Identity()
        if stride != 1 or c_in != c_out:
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class DressaNet(nn.Module):
    def __init__(self, n_classes: dict, widths=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 2, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU())
        layers, c_in = [], widths[0]
        for i, w in enumerate(widths):
            stride = 1 if i == 0 else 2
            layers += [BasicBlock(c_in, w, stride), BasicBlock(w, w, 1)]
            c_in = w
        self.features = nn.Sequential(*layers)
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({name: nn.Linear(c_in, n) for name, n in n_classes.items()})

    def forward(self, x):
        x = self.features(self.stem(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        x = self.dropout(x)
        # Tuple (bukan dict) supaya urutan output jelas saat export ONNX
        return tuple(self.heads[name](x) for name in HEADS)


N_CLASSES = {h: len(CLASSES[h]) for h in HEADS}
model = DressaNet(N_CLASSES, dropout=CONFIG["dropout"]).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"DressaNet: {n_params / 1e6:.2f} juta parameter, semua dimulai acak (dilatih dari nol)")
with torch.no_grad():
    outs = model(torch.zeros(2, 3, CONFIG["img_size"], CONFIG["img_size"], device=device))
print({h: tuple(o.shape) for h, o in zip(HEADS, outs)})

## 6. Training

**Loss function** mengukur seberapa salah tebakan model. Untuk klasifikasi, yang dipakai adalah **cross-entropy**: nilainya kecil kalau model memberi probabilitas tinggi ke kelas yang benar. Total loss adalah jumlah loss ketiga head setelah dikali bobotnya:

$$L = 1.0 \cdot L_{\text{item}} + 0.5 \cdot L_{\text{style}} + 0.5 \cdot L_{\text{season}}$$

Tambahan pada loss:
- `weight`: class weight untuk kelas kecil (lihat bagian 3). Rumusnya $w_c \propto 1/\sqrt{n_c}$, yaitu lebih lunak daripada $1/n_c$ supaya kelas kecil tidak terlalu dominan.
- `label_smoothing=0.1`: target "100% benar" diganti menjadi sekitar 90%, sehingga model tidak terlalu yakin diri. Cara ini biasanya membuat model lebih baik di data baru.
- `ignore_index=-1`: label yang tidak diketahui dilewati.

**Satu langkah training:**
1. *Forward*: input dimasukkan ke model, menghasilkan logit.
2. Hitung loss.
3. *Backward* (`loss.backward()`): PyTorch menghitung gradient, yaitu arah perubahan setiap bobot yang menurunkan loss.
4. **Optimizer** (`AdamW`) memperbarui bobot ke arah itu.

**OneCycle LR schedule**: learning rate naik pelan di awal (warm-up), lalu turun sampai hampir nol di akhir. Cara ini biasanya lebih cepat dan hasilnya lebih baik dibanding LR tetap.

**Mixed precision (AMP)** menghitung sebagian operasi dengan angka 16-bit di GPU. Training jadi sekitar 2× lebih cepat, hampir tanpa kehilangan akurasi.

Di akhir setiap epoch, model dievaluasi di validation set. Bobot dengan **macro-F1 item_type tertinggi** disimpan sebagai model terbaik.

In [ ]:
def class_weights(labels, n):
    counts = np.bincount(labels[labels >= 0], minlength=n).astype(float)
    w = 1.0 / np.sqrt(np.maximum(counts, 1))
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=device)

criteria = {
    h: nn.CrossEntropyLoss(
        weight=class_weights(Y["train"][:, i], N_CLASSES[h]) if h == "item_type" else None,
        label_smoothing=CONFIG["label_smoothing"],
        ignore_index=-1,
    )
    for i, h in enumerate(HEADS)
}

def compute_loss(outputs, yb):
    total = 0.0
    for i, h in enumerate(HEADS):
        if (yb[:, i] >= 0).any():  # semua label di batch tidak diketahui -> lewati head ini
            total = total + CONFIG["loss_weights"][h] * criteria[h](outputs[i].float(), yb[:, i])
    return total

@torch.no_grad()
def predict(model, loader):
    model.eval()  # matikan dropout, BatchNorm pakai statistik tetap
    preds, labels, probs = {h: [] for h in HEADS}, [], {h: [] for h in HEADS}
    for xb, yb in loader:
        outs = model(to_input(xb, train=False))
        for h, o in zip(HEADS, outs):
            p = o.float().softmax(1)
            probs[h].append(p.cpu()); preds[h].append(p.argmax(1).cpu())
        labels.append(yb)
    labels = torch.cat(labels).numpy()
    return ({h: torch.cat(preds[h]).numpy() for h in HEADS}, labels, {h: torch.cat(probs[h]).numpy() for h in HEADS})

def scores(preds, labels):
    out = {}
    for i, h in enumerate(HEADS):
        m = labels[:, i] >= 0
        out[h] = {
            "acc": float((preds[h][m] == labels[m, i]).mean()),
            "macro_f1": float(f1_score(labels[m, i], preds[h][m], average="macro")),
        }
    return out

def train_model(model, epochs, lr, tag):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CONFIG["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(loaders["train"]))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best_f1, history = -1.0, []
    ckpt = WORK_DIR / f"{tag}_best.pt"
    for epoch in range(1, epochs + 1):
        model.train()
        t0, run_loss, n = time.time(), 0.0, 0
        for xb, yb in loaders["train"]:
            x, yb = to_input(xb, train=True), yb.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                outputs = model(x)
            loss = compute_loss(outputs, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            run_loss += loss.item() * len(xb); n += len(xb)
        preds, labels, _ = predict(model, loaders["val"])
        s = scores(preds, labels)
        history.append({"epoch": epoch, "train_loss": run_loss / n, **{f"val_{h}_{k}": v for h in HEADS for k, v in s[h].items()}})
        mark = ""
        if s["item_type"]["macro_f1"] > best_f1:
            best_f1 = s["item_type"]["macro_f1"]
            torch.save(model.state_dict(), ckpt)
            mark = "  <- terbaik"
        print(f"[{tag}] epoch {epoch:2d}/{epochs} loss {run_loss / n:.3f} | val item F1 {s['item_type']['macro_f1']:.3f} "
              f"acc {s['item_type']['acc']:.3f} | style F1 {s['style']['macro_f1']:.3f} | season F1 {s['season']['macro_f1']:.3f} "
              f"| {time.time() - t0:.0f}s{mark}")
    model.load_state_dict(torch.load(ckpt, map_location=device))
    return pd.DataFrame(history)

history = train_model(model, CONFIG["epochs"], CONFIG["lr"], tag="dressanet")

### Kurva training

Cara membaca grafik ini:
- **Train loss terus turun dan val F1 terus naik**: model masih belajar. Coba tambah epoch.
- **Train loss turun tapi val F1 mendatar atau turun**: tanda **overfitting**, yaitu model menghafal data training. Tambah augmentasi, dropout, atau weight decay.
- **Keduanya buruk sejak awal**: *underfitting*. Model terlalu kecil atau learning rate tidak cocok.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history["epoch"], history["train_loss"]); axes[0].set_title("Train loss"); axes[0].set_xlabel("epoch")
for h in HEADS:
    axes[1].plot(history["epoch"], history[f"val_{h}_macro_f1"], label=h)
axes[1].set_title("Validation macro-F1"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Evaluasi di test set

Bagian ini dijalankan **sekali** dengan model terbaik. Angka di sini yang masuk ke laporan.

Arti metrik:
- **Accuracy**: persentase tebakan yang benar.
- **Precision** kelas X: dari semua yang ditebak X, berapa persen yang benar-benar X.
- **Recall** kelas X: dari semua X yang sebenarnya, berapa persen yang berhasil ditemukan.
- **F1**: rata-rata harmonik precision dan recall.
- **Macro-F1**: rata-rata F1 semua kelas, dengan bobot sama untuk setiap kelas.
- **Confusion matrix**: baris adalah kelas sebenarnya, kolom adalah tebakan. Kotak di luar diagonal menunjukkan kelas mana yang sering tertukar.

In [ ]:
test_preds, test_labels, test_probs = predict(model, loaders["test"])
TEST_SCORES = scores(test_preds, test_labels)
print(pd.DataFrame(TEST_SCORES).T.round(3))
for i, h in enumerate(HEADS):
    m = test_labels[:, i] >= 0
    print(f"\n=== {h} ===")
    print(classification_report(test_labels[m, i], test_preds[h][m], labels=range(N_CLASSES[h]),
                                target_names=CLASSES[h], digits=3, zero_division=0))

In [ ]:
def plot_confusion(h, figsize):
    i = HEADS.index(h)
    m = test_labels[:, i] >= 0
    cm = confusion_matrix(test_labels[m, i], test_preds[h][m], labels=range(N_CLASSES[h]), normalize="true")
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_CLASSES[h]), CLASSES[h], rotation=70, ha="right")
    ax.set_yticks(range(N_CLASSES[h]), CLASSES[h])
    ax.set_xlabel("tebakan"); ax.set_ylabel("sebenarnya"); ax.set_title(f"Confusion matrix {h} (dinormalisasi per baris)")
    plt.colorbar(im); plt.tight_layout(); plt.show()

plot_confusion("item_type", (11, 9))
plot_confusion("style", (5, 4))

### Contoh kesalahan

Melihat foto yang salah ditebak sering lebih berguna daripada angka: apakah labelnya yang salah, kelasnya memang mirip (misalnya top vs t-shirt), atau model kurang data?

In [ ]:
wrong = np.where(test_preds["item_type"] != test_labels[:, 0])[0]
conf = test_probs["item_type"][wrong].max(1)
show = wrong[np.argsort(-conf)][:16]  # kesalahan yang paling "yakin"
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for ax, j in zip(axes.flat, show):
    ax.imshow(X["test"][j])
    ax.set_title(f"asli: {CLASSES['item_type'][test_labels[j, 0]]}\ntebak: {CLASSES['item_type'][test_preds['item_type'][j]]}", fontsize=8)
    ax.axis("off")
plt.suptitle(f"{len(wrong)} dari {len(test_labels)} foto test salah item_type"); plt.tight_layout(); plt.show()

### Coba dengan foto sendiri (opsional)

Upload foto pakaian. Hasil paling baik didapat dari foto yang **background-nya sudah dihapus** (misalnya file `cutout.png` dari Dressa), karena dataset training berlatar putih. Foto dengan latar ramai kemungkinan besar ditebak salah. Perbedaan seperti ini disebut *domain gap*, dan menarik untuk dibahas di laporan.

In [ ]:
def predict_image(img: Image.Image):
    x = torch.from_numpy(np.asarray(preprocess(img, CONFIG["img_size"]))).permute(2, 0, 1)[None]
    model.eval()
    with torch.no_grad():
        outs = model(to_input(x, train=False))
    return {h: sorted(zip(CLASSES[h], o.float().softmax(1)[0].tolist()), key=lambda t: -t[1])[:3] for h, o in zip(HEADS, outs)}

if IN_COLAB:
    from google.colab import files
    for name, content in files.upload().items():
        import io
        img = Image.open(io.BytesIO(content))
        plt.figure(figsize=(3, 3)); plt.imshow(img); plt.axis("off"); plt.show()
        for h, top in predict_image(img).items():
            print(f"{h:10s}", ", ".join(f"{c} {p:.0%}" for c, p in top))

## 8. Export ke ONNX

**ONNX** adalah format standar untuk menyimpan model terlatih. File `.onnx` bisa dijalankan dengan **ONNX Runtime** tanpa PyTorch. `ml-engine` Dressa sudah memakai ONNX Runtime, jadi model ini bisa langsung dipasang di sana.

Sel ini menghasilkan dua file:
- `dressa_attr.onnx`: model (input `image` berukuran `(N, 3, 80, 80)` yang sudah dinormalisasi; output `item_type`, `style`, `season` berupa logit).
- `dressa_attr_labels.json`: daftar kelas, ukuran input, mean/std, pemetaan ke kategori Dressa, dan skor test.

Setelah itu output ONNX dibandingkan dengan output PyTorch untuk memastikan hasil export-nya sama.

In [ ]:
import onnxruntime as ort

model.eval().cpu()
dummy = torch.randn(2, 3, CONFIG["img_size"], CONFIG["img_size"])
onnx_path = WORK_DIR / "dressa_attr.onnx"
try:
    # Exporter baru (default sejak PyTorch 2.9). Dimensi batch dibuat dinamis.
    torch.onnx.export(
        model, (dummy,), str(onnx_path), input_names=["image"], output_names=HEADS,
        dynamic_shapes=({0: torch.export.Dim("batch", min=1, max=1024)},), dynamo=True,
        external_data=False,  # simpan bobot di dalam satu file .onnx
    )
except Exception as e:
    print("Exporter baru gagal, pakai exporter lama:", repr(e)[:200])
    torch.onnx.export(
        model, (dummy,), str(onnx_path), input_names=["image"], output_names=HEADS,
        dynamic_axes={"image": {0: "batch"}, **{h: {0: "batch"} for h in HEADS}}, opset_version=18, dynamo=False,
    )

meta = {
    "model": "DressaNet",
    "trained_from_scratch": True,
    "img_size": CONFIG["img_size"],
    "mean": MEAN, "std": STD,
    "preprocess": "flatten alpha on white, pad to square with white, resize bilinear, /255, (x-mean)/std, NCHW float32",
    "heads": HEADS,
    "classes": CLASSES,
    "item_type_to_category": ITEM_TYPE_TO_CATEGORY,
    "test_scores": TEST_SCORES,
    "config": CONFIG,
    "n_train": len(train_df),
}
labels_path = WORK_DIR / "dressa_attr_labels.json"
labels_path.write_text(json.dumps(meta, indent=2))

# Verifikasi: ONNX Runtime vs PyTorch pada 64 gambar test
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
xb = torch.from_numpy(X["test"][:64]).permute(0, 3, 1, 2).float() / 255
xb = (xb - torch.tensor(MEAN).view(1, 3, 1, 1)) / torch.tensor(STD).view(1, 3, 1, 1)
with torch.no_grad():
    torch_out = model(xb)
onnx_out = sess.run(None, {"image": xb.numpy()})
for h, t, o in zip(HEADS, torch_out, onnx_out):
    print(f"{h:10s} max selisih {np.abs(t.numpy() - o).max():.2e} | tebakan sama: {(t.argmax(1).numpy() == o.argmax(1)).mean():.0%}")
print(f"\n{onnx_path.name}: {onnx_path.stat().st_size / 1e6:.1f} MB")
model.to(device)

if IN_COLAB:
    from google.colab import files
    files.download(str(onnx_path))
    files.download(str(labels_path))

Simpan kedua file tersebut ke `services/ml-engine/weights/` di project Dressa. Folder itu sudah di-gitignore. Pemasangan modelnya ke `ml-engine` adalah langkah berikutnya.

## 9. (Opsional) Pembanding: transfer learning

Supaya laporan punya pembanding, bagian ini melatih **ResNet18 pretrained ImageNet** dengan data, augmentasi, dan loss yang sama. Model ini sudah "melihat" 1,2 juta foto sebelum belajar pakaian, jadi biasanya lebih akurat. Pertanyaan yang menarik untuk laporan: **seberapa jauh selisihnya** dengan model dari nol, dan berapa biayanya (jumlah parameter, ukuran file, kecepatan)?

Gambar diperbesar ke 160×160 karena ResNet18 dilatih dengan gambar yang lebih besar. Ubah `CONFIG["run_pretrained_baseline"] = False` untuk melewati bagian ini.

In [ ]:
if CONFIG["run_pretrained_baseline"]:
    import torchvision

    class PretrainedNet(nn.Module):
        def __init__(self, n_classes, dropout=0.3):
            super().__init__()
            backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
            backbone.fc = nn.Identity()
            self.backbone, self.dropout = backbone, nn.Dropout(dropout)
            self.heads = nn.ModuleDict({h: nn.Linear(512, n) for h, n in n_classes.items()})

        def forward(self, x):
            x = F.interpolate(x, size=160, mode="bilinear", align_corners=False)
            x = self.dropout(self.backbone(x))
            return tuple(self.heads[h](x) for h in HEADS)

    baseline = PretrainedNet(N_CLASSES, CONFIG["dropout"]).to(device)
    base_hist = train_model(baseline, CONFIG["baseline_epochs"], lr=1e-3, tag="resnet18_pretrained")
    base_scores = scores(*predict(baseline, loaders["test"])[:2])

    def speed(m, n=256):
        m.eval(); x = torch.randn(n, 3, CONFIG["img_size"], CONFIG["img_size"], device=device)
        with torch.no_grad():
            m(x); torch.cuda.synchronize() if device.type == "cuda" else None
            t = time.time(); m(x); torch.cuda.synchronize() if device.type == "cuda" else None
        return (time.time() - t) / n * 1000

    rows = []
    for name, m, s in [("DressaNet (dari nol)", model, TEST_SCORES), ("ResNet18 (pretrained)", baseline, base_scores)]:
        rows.append({"model": name, "params (juta)": sum(p.numel() for p in m.parameters()) / 1e6,
                     "ms/gambar": speed(m), **{f"{h} F1": s[h]["macro_f1"] for h in HEADS},
                     "item acc": s["item_type"]["acc"]})
    print(pd.DataFrame(rows).round(3).to_string(index=False))

## Ide eksperimen untuk laporan

- **Ablation**: latih ulang tanpa augmentasi, tanpa class weight, atau tanpa label smoothing, lalu bandingkan macro-F1. Dari situ terlihat teknik mana yang benar-benar berpengaruh.
- **Ukuran model**: ubah `widths` (misalnya `(16, 32, 64, 128)` atau `(64, 128, 256, 512)`) lalu lihat pengaruhnya ke akurasi, jumlah parameter, dan kecepatan.
- **Single-task vs multi-task**: latih model khusus `item_type` (set bobot loss head lain ke 0), lalu bandingkan.
- **Domain gap**: kumpulkan 30–50 foto baju sendiri yang sudah di-cutout lewat Dressa, beri label manual, lalu ukur akurasinya. Bandingkan dengan akurasi di test set Kaggle.